# Chapter 4: Vector Spaces & Orthogonality

Every matrix has **four fundamental subspaces** — understanding them reveals the full picture of $Ax = b$:

| Subspace | Symbol | Dimension |
|---|---|---|
| Column space (range) | $\mathcal{C}(A)$ | $r = \text{rank}(A)$ |
| Row space | $\mathcal{C}(A^T)$ | $r$ |
| Null space (kernel) | $\mathcal{N}(A)$ | $n - r$ |
| Left null space | $\mathcal{N}(A^T)$ | $m - r$ |

## Learning Objectives
- Understand vector space axioms and subspaces
- Compute column space, null space, and their dimensions
- Apply the rank-nullity theorem
- Perform Gram-Schmidt orthogonalisation
- Compute orthogonal projections


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg as sla
from mpl_toolkits.mplot3d import Axes3D

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=4, suppress=True)


## Vector Spaces: Formal Definition

A set $V$ is a **vector space** if it is closed under addition and scalar multiplication, and contains the zero vector:

$$\forall\, u, v \in V,\; \alpha, \beta \in \mathbb{R} :\quad \alpha u + \beta v \in V$$

**Subspace test** (three conditions):
1. $\mathbf{0} \in V$
2. $u, v \in V \Rightarrow u + v \in V$
3. $u \in V,\; \alpha \in \mathbb{R} \Rightarrow \alpha u \in V$


In [ ]:
np.random.seed(7)

# V = {(x,y) : x + y = 0}  -- IS a subspace
def in_V(v): return np.isclose(v[0] + v[1], 0)

# Test 200 random linear combinations
passes = 0
for _ in range(200):
    a, b = np.random.randn(2)
    combo = a*np.array([1.,-1.]) + b*np.array([2.,-2.])
    passes += int(in_V(combo))
print(f'V (x+y=0): {passes}/200 random combos stay in V')

# W = {(x,y) : x + y = 1}  -- NOT a subspace (doesn't contain 0)
zero_in_W = np.isclose(0 + 0, 1)
print(f'W (x+y=1): contains zero? {zero_in_W} => NOT a subspace')


## Column Space (Range)

$$\mathcal{C}(A) = \{Ax : x \in \mathbb{R}^n\} = \text{all linear combinations of columns of } A$$

This is the set of all **achievable right-hand sides** $b$ for which $Ax = b$ has a solution.


In [ ]:
A = np.array([[1, 0],
              [0, 1],
              [1, 1]], dtype=float)  # 3×2, column space = plane in R^3

# Sample many vectors in the column space
np.random.seed(0)
coeffs = np.random.randn(2, 500)
points = A @ coeffs  # all linear combinations

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(*points, alpha=0.15, s=5, c='steelblue')

# Plot the two columns
for col, clr, lbl in zip(A.T, ['red','green'], ['col 1','col 2']):
    ax.quiver(0,0,0, *col, color=clr, lw=3, arrow_length_ratio=0.1, label=lbl)

ax.set(title='Column Space of A (plane in ℝ³)',
       xlabel='x', ylabel='y', zlabel='z')
ax.legend(); plt.tight_layout(); plt.show()
print("Column space dimension (rank):", np.linalg.matrix_rank(A))


## Null Space (Kernel)

$$\mathcal{N}(A) = \{x : Ax = \mathbf{0}\}$$

The null space contains all vectors that $A$ **"kills"** (maps to zero).

Fundamental theorem: **null space $\perp$ row space**.


In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]], dtype=float)

# Compute null space using SVD
ns = sla.null_space(A)
print("Null space basis:\n", ns)
print("A @ null_vector ≈ 0?\n", np.round(A @ ns, 10))

rank = np.linalg.matrix_rank(A)
nullity = A.shape[1] - rank
print(f"\nrank={rank}, nullity={nullity}, n={A.shape[1]}")
print("rank + nullity == n?", rank + nullity == A.shape[1])


## Rank-Nullity Theorem

For any $A \in \mathbb{R}^{m \times n}$:

$$\text{rank}(A) + \text{nullity}(A) = n$$

where $\text{rank}(A) = \dim(\mathcal{C}(A))$ and $\text{nullity}(A) = \dim(\mathcal{N}(A))$.

Beautiful symmetry: rank = dim(column space) = dim(row space).


In [ ]:
def verify_rank_nullity(A):
    m, n = A.shape
    r = np.linalg.matrix_rank(A)
    nullity = n - r
    ns = sla.null_space(A)
    print(f"Shape {m}×{n}: rank={r}, nullity={nullity}, "
          f"null_space_dim={ns.shape[1] if ns.ndim==2 else 0}, "
          f"theorem holds: {r + nullity == n}")

verify_rank_nullity(np.random.randn(3, 5))        # rank-deficient (likely rank 3)
verify_rank_nullity(np.array([[1,2,3],[2,4,6.],[0,0,1]]))  # rank 2
verify_rank_nullity(np.eye(4))                     # full rank


## Basis

A **basis** for a vector space $V$ is a set of vectors that is:
1. **Linearly independent** — no vector is a combination of the others
2. **Spanning** — every vector in $V$ can be written as a linear combination

The **dimension** of $V$ = number of basis vectors (always the same, regardless of which basis you choose).

Standard basis of $\mathbb{R}^n$: $e_1 = (1,0,\ldots,0)^T,\; e_2=(0,1,0,\ldots)^T,\;\ldots$


In [ ]:
A = np.array([[1, 2, 3, 4],
              [2, 4, 5, 6],
              [3, 6, 7, 8]], dtype=float)

# Find a basis for column space via QR with pivoting
Q, R, P = sla.qr(A, pivoting=True)
rank = np.linalg.matrix_rank(A)
basis_cols = A[:, P[:rank]]  # independent columns

print("Rank:", rank)
print("Basis for column space (pivot columns):\n", basis_cols)
print("These columns span the same space as A:",
      np.allclose(np.linalg.matrix_rank(basis_cols),
                  np.linalg.matrix_rank(np.hstack([basis_cols, A]))))


## Orthogonality

Vectors $u$ and $v$ are **orthogonal** if their dot product is zero:

$$u \perp v \Leftrightarrow u \cdot v = u^T v = 0$$

An **orthonormal** set satisfies $q_i \cdot q_j = \delta_{ij}$ (Kronecker delta).
An **orthonormal matrix** $Q$ satisfies $Q^T Q = I$.


In [ ]:
# Standard basis is orthonormal
e1, e2, e3 = np.eye(3)
print("e1·e2 =", e1 @ e2, "  e1·e1 =", e1 @ e1)

# Create an orthonormal set from random vectors using QR
np.random.seed(3)
M = np.random.randn(4, 4)
Q, R = np.linalg.qr(M)
print("\nQ^T Q =\n", np.round(Q.T @ Q, 10))
print("Q is orthonormal:", np.allclose(Q.T @ Q, np.eye(4)))


## Gram-Schmidt Orthogonalisation

Convert any linearly independent set $\{v_1, \ldots, v_k\}$ to orthonormal $\{q_1, \ldots, q_k\}$:

$$\tilde{q}_k = v_k - \sum_{i=1}^{k-1} (v_k \cdot q_i)\, q_i, \qquad q_k = \frac{\tilde{q}_k}{\|\tilde{q}_k\|}$$

This is exactly what `np.linalg.qr` does internally.


In [ ]:
def gram_schmidt(V):
    # Orthonormalise columns of V.
    Q = np.zeros_like(V, dtype=float)
    for k in range(V.shape[1]):
        q = V[:, k].astype(float).copy()
        for i in range(k):
            q -= (q @ Q[:, i]) * Q[:, i]  # subtract projection
        Q[:, k] = q / np.linalg.norm(q)
    return Q

np.random.seed(5)
V = np.random.randn(4, 3)

Q_scratch = gram_schmidt(V)
Q_numpy, R_numpy = np.linalg.qr(V)

print("From scratch Q^T Q:\n", np.round(Q_scratch.T @ Q_scratch, 10))
print("NumPy QR  Q^T Q:\n",    np.round(Q_numpy.T @ Q_numpy, 10))
# Columns may differ by sign, but they span the same space
print("Same column space?", np.allclose(
    np.linalg.matrix_rank(np.hstack([Q_scratch, Q_numpy])), 3))


## Orthogonal Projection

Project vector $b$ onto the subspace spanned by columns of $A$:

$$\hat{b} = A(A^T A)^{-1} A^T b = Pb$$

The **projection matrix** $P = A(A^T A)^{-1}A^T$ is symmetric and idempotent ($P^2 = P$).
The **residual** $b - \hat{b}$ is orthogonal to every column of $A$: $A^T(b - \hat{b}) = 0$.


In [ ]:
A_proj = np.array([[1, 0],
                   [0, 1],
                   [1, 1]], dtype=float)
b_vec  = np.array([1.0, 2.0, 0.0])

# Projection matrix P = A(A^T A)^{-1} A^T
P = A_proj @ np.linalg.inv(A_proj.T @ A_proj) @ A_proj.T
b_hat = P @ b_vec
residual = b_vec - b_hat

print("b     =", b_vec)
print("b_hat =", b_hat)
print("residual =", residual)
print("A^T @ residual ≈ 0?", np.allclose(A_proj.T @ residual, 0))
print("P is idempotent (P²=P)?", np.allclose(P @ P, P))

# 2D visualization
fig, ax = plt.subplots(figsize=(5, 5))
origin = np.zeros(2)
# Project onto first column only for 2D viz
a_2d = A_proj[:2, 0]
b_2d = b_vec[:2]
p_2d = (a_2d @ b_2d)/(a_2d @ a_2d) * a_2d
for v, c, lbl in [(b_2d,'blue','b'), (p_2d,'red',r'$\hat{b}$'), (b_2d-p_2d,'green',r'$b-\hat{b}$')]:
    ax.annotate('', xy=v, xytext=origin if lbl!= r'$b-\hat{b}$' else p_2d,
                arrowprops=dict(arrowstyle='->', color=c, lw=2))
    ax.text(*(v*0.6 if lbl!= r'$b-\hat{b}$' else p_2d+v*0.3), lbl, color=c, fontsize=13)
ax.set(xlim=(-0.5,2), ylim=(-0.5,2.5), aspect='equal',
       title='Orthogonal Projection (2D slice)'); ax.grid(True)
plt.tight_layout(); plt.show()


## Summary

| Concept | Key Formula | NumPy |
|---|---|---|
| Null space | $\mathcal{N}(A) = \{x: Ax=0\}$ | `scipy.linalg.null_space(A)` |
| Rank | $\dim(\mathcal{C}(A))$ | `np.linalg.matrix_rank(A)` |
| Rank-Nullity | $r + \text{nullity} = n$ | verify manually |
| Gram-Schmidt | iterative orthogonalisation | `np.linalg.qr(A)` |
| Projection | $\hat{b} = A(A^TA)^{-1}A^Tb$ | matrix formula |

**Next:** Chapter 5 — Eigenvalues & Eigenvectors: the "natural coordinates" of a matrix.
